# MirrorTopology Step 1 — Phase D-3a：12 位置 base 共分散＋PC-1（family 単位；v0.2）
受入れ済み D-1 生成器（A11 cell 4 逐語・pinned CMBtopology `0cc65e34`）で，1 family の**新 27 配置の base 共分散**を生成・intake し，PC-1（clone を x₀_H1=M b−T で独立生成，bridge 求積の D(M)，rel=‖C1−DC0Dᵀ‖_F/‖target‖_F，match rel<1e-5；case 表どおり）と第 1 波 anchor 診断（base は D-1 の固定 SHA）を実行する。出力は Drive；registry と PC-1 結果は監査用 zip。**family を 1 つずつ別 run**（E2／E7 ≈8 h，E8 ≈11 h の見込み；保証ではない；Colab の session 上限に掛かる場合は `SIZE_FILTER` で **正式 size partition**（当該 size の新配置と旧 anchor case を含む；family PASS は登録 partition の集約 `aggregate_partitions` で構成）。label なし・較正なし。

In [ ]:
# --- 0. OUTER LAUNCHER LOCK (the only editable cell)
REPO_URL = 'https://github.com/tsujikeita/mirror-topology.git'
REPO_COMMIT = '<full 40-hex commit of the verification target>'
EXPECTED_INVENTORY_SHA256 = '<sha256 of engine/phaseB/B2_completion_inventory.json inside that commit>'
FAMILY = 'E2'                      # E2 / E7 / E8 ; one family per run
SIZE_FILTER = ''                   # '' = whole family; or 'L1.00' etc. = FORMAL size partition (includes that size's anchor cases; each partition is a separate attempt)
DRIVE_DIR = '/content/drive/MyDrive/MirrorTopology_D3'
WITH_H2 = False                    # True adds the H2 alternative clone per case (diagnostic; doubles clone cost)
LAUNCHER_ID = 'MirrorTopology_Step1_D3a_covgen_v0.2'


In [ ]:
# --- 1. fresh scratch checkout at C; inventory bound; pins+script bound; Phase C packet at the same commit
import subprocess, sys, os, json, hashlib, shutil, time, re
sha=lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT) and re.fullmatch(r'[0-9a-f]{64}', EXPECTED_INVENTORY_SHA256) and FAMILY in ('E2','E7','E8'), 'launcher lock not filled'
from google.colab import drive; drive.mount('/content/drive'); os.makedirs(DRIVE_DIR, exist_ok=True)
RUN=f'{DRIVE_DIR}/{FAMILY}{("_"+SIZE_FILTER) if SIZE_FILTER else ""}_{time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())}'; SCRATCH='/content/d3_scratch'; OUT=f'{RUN}/out'; os.makedirs(OUT)
shutil.rmtree(SCRATCH, ignore_errors=True); subprocess.run(['git','clone','-q',REPO_URL,SCRATCH],check=True); subprocess.run(['git','-C',SCRATCH,'checkout','-q',REPO_COMMIT],check=True)
head=subprocess.check_output(['git','-C',SCRATCH,'rev-parse','HEAD']).decode().strip(); assert head==REPO_COMMIT, head; assert subprocess.check_output(['git','-C',SCRATCH,'status','--porcelain']).decode().strip()==''
MT=SCRATCH; PHASEB=f'{MT}/engine/phaseB'; PHASEC=f'{MT}/phaseC'; INV=f'{PHASEB}/B2_completion_inventory.json'; inv_sha=sha(INV); assert inv_sha==EXPECTED_INVENTORY_SHA256, inv_sha
inv=json.load(open(INV)); PINS=f'{PHASEB}/d/d3_pins.json'; SCRIPT=f'{PHASEB}/d/d3_covgen.py'; assert sha(PINS)==inv['d_sha256']['d/d3_pins.json'] and sha(SCRIPT)==inv['d_sha256']['d/d3_covgen.py']
pins=json.load(open(PINS)); assert pins['schema']=='d3_pins_v1' and pins['engine_version']==inv['engine_version'] and sha(f'{PHASEC}/PACKET_INVENTORY.json')==pins['phaseC_inventory_sha256']
lock=dict(launcher_id=LAUNCHER_ID, repo_url=REPO_URL, commit=head, inventory_sha256=inv_sha, pins_sha256=sha(PINS), engine_version=inv['engine_version'], family=FAMILY, size_filter=SIZE_FILTER or None, with_h2=bool(WITH_H2), run_dir=RUN); json.dump(lock, open(f'{OUT}/launcher_lock.json','w'), indent=1); print(lock)


In [ ]:
# --- 2. pinned CMBtopology + registered environment (script re-verifies live versions / pools as a HARD gate)
CT='/content/CMBtopology_pinned'; shutil.rmtree(CT, ignore_errors=True); subprocess.run(['git','clone','-q',pins['cmbtopology']['url'],CT],check=True); subprocess.run(['git','-C',CT,'checkout','-q',pins['cmbtopology']['commit']],check=True)
ex=pins['environment']; subprocess.run([sys.executable,'-m','pip','install','-q',f"numpy=={ex['numpy']}",f"scipy=={ex['scipy']}",f"healpy=={ex['healpy']}",f"camb=={ex['camb']}",f"pot=={ex['pot']}",'threadpoolctl','numba','quaternionic','spherical','tqdm','pandas','matplotlib'],check=True)
os.environ['OPENBLAS_NUM_THREADS']='2'; os.environ['OMP_NUM_THREADS']='2'
import platform, importlib; live={k: importlib.import_module({'pot':'ot'}.get(k,k)).__version__ for k in ('numpy','scipy','healpy','camb','pot')}; live['python']=platform.python_version(); mism={k:(live[k],ex[k]) for k in live if live[k]!=ex[k]}; assert not mism, f'environment lock failed (launcher pre-check): {mism}'
print('environment ok', live)


In [ ]:
# --- 3. generation + PC-1 (long); output goes straight to Drive
import json as _j
DO=f'{OUT}/d3'; args=[sys.executable,SCRIPT,'--mt',MT,'--phaseb',PHASEB,'--phasec',PHASEC,'--ct',CT,'--out',DO,'--family',FAMILY,'--profile','production_official']+(['--with-h2'] if WITH_H2 else [])+(['--sizes',SIZE_FILTER] if SIZE_FILTER else [])
rc=subprocess.run(args,capture_output=True,text=True); open(f'{OUT}/launcher_script_stdout.txt','w').write(rc.stdout); open(f'{OUT}/launcher_script_stderr.txt','w').write(rc.stderr); print(rc.stdout[-3000:])
try: rm=_j.load(open(f'{DO}/d3_run_manifest.json'))
except Exception as _ex: rm=dict(D3_PASS=False, stage='RECORD_MISSING_OR_INVALID', failures=[f'run manifest unreadable: {_ex!r}', f'script returncode {rc.returncode}'], launcher_fallback=True, stderr_tail=rc.stderr[-2000:])   # machine-readable launcher failure record
print('script rc', rc.returncode, 'D3_PASS', rm.get('D3_PASS'), rm.get('failures'), 'bases', rm.get('n_bases'), 'cases', rm.get('n_cases'), 'partition', rm.get('partition'))


In [ ]:
# --- 4. final record + audit zip (covariance .npy stay on Drive; registry, PC-1 results, env lock, manifests, logs are zipped)
final=dict(launcher=lock, D3_PASS=bool(rc.returncode==0 and rm.get('D3_PASS') is True and not rm.get('launcher_fallback')), family=FAMILY, size_filter=SIZE_FILTER or None, stages=dict(script_returncode=rc.returncode, script_pass=rm.get('D3_PASS'), stage=rm.get('stage'), gates=rm.get('gates'), failures=rm.get('failures'), launcher_fallback=bool(rm.get('launcher_fallback')), n_bases=rm.get('n_bases'), n_cases=rm.get('n_cases'), pc1_status=rm.get('pc1_status'), seconds=rm.get('seconds'), source=rm.get('source')))
json.dump(final, open(f'{OUT}/d3_final_record.json','w'), indent=1); print(json.dumps({k:final[k] for k in ('D3_PASS','family','size_filter')}, indent=1), 'run dir:', RUN)
import zipfile; zp=f'/content/d3_{FAMILY}{("_"+SIZE_FILTER) if SIZE_FILTER else ""}_{REPO_COMMIT[:12]}_audit.zip'
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:
    for d,_,fs in os.walk(OUT):
        for f in fs:
            if not f.endswith('.npy'): z.write(os.path.join(d,f), os.path.relpath(os.path.join(d,f), OUT))
from google.colab import files; print(zp, os.path.getsize(zp)); files.download(zp)
